In [10]:
!git clone https://github.com/nirajj12/Bird-s-Eye-View-BEV-2D-Occupancy.git

Cloning into 'Bird-s-Eye-View-BEV-2D-Occupancy'...
remote: Enumerating objects: 7996, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7996 (delta 0), reused 3 (delta 0), pack-reused 7992 (from 2)
Receiving objects: 100% (7996/7996), 41.72 MiB | 24.01 MiB/s, done.
Resolving deltas: 100% (1263/1263), done.


In [11]:
import subprocess
subprocess.run(['pip', 'install', 'numpy==1.26.4', '--force-reinstall', '-q'], check=True)
subprocess.run(['pip', 'install', 'scipy==1.11.4', '--force-reinstall', '-q'], check=True)
subprocess.run(['pip', 'install', 'scikit-learn==1.3.2', '--force-reinstall', '-q'], check=True)
print("Done!")

Done!


In [12]:
!pip install -q nuscenes-devkit==1.1.11 --no-deps
!pip install -q pyquaternion descartes fire tqdm einops timm Pillow matplotlib


In [13]:
import numpy, torch
print("numpy :", numpy.__version__)
print("torch :", torch.__version__)
print("GPU   :", torch.cuda.is_available())

from nuscenes.nuscenes import NuScenes
print("nuscenes OK ✓")

numpy : 1.26.4
torch : 2.10.0+cu128
GPU   : True
nuscenes OK ✓


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os, sys
os.chdir('/content')
!git clone https://github.com/nirajj12/Bird-s-Eye-View-BEV-2D-Occupancy.git
os.chdir('/content/Bird-s-Eye-View-BEV-2D-Occupancy')
sys.path.insert(0, '/content/Bird-s-Eye-View-BEV-2D-Occupancy')

print("Repo ready ✓")
print("Files:", os.listdir('.'))

fatal: destination path 'Bird-s-Eye-View-BEV-2D-Occupancy' already exists and is not an empty directory.
Repo ready ✓
Files: ['train.py', 'logger', 'checkpoints', 'data', '.git', 'Bird-s-Eye-View-BEV-2D-Occupancy', 'notebooks', 'config', 'requirements.txt', '.gitignore', 'eval.py', 'utils', 'models', 'exception', 'results']


In [16]:
import os

bev_folder = '/content/drive/MyDrive/BEV_PROJECT'

for root, dirs, files in os.walk(bev_folder):
    level = root.replace(bev_folder, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    for f in sorted(files):
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1e6
        print(f"{indent}  📄 {f}  ({size:.1f} MB)")

📁 BEV_PROJECT/
  📄 best_model.pth  (137.7 MB)
  📄 epoch_01.pth  (137.7 MB)
  📄 epoch_02.pth  (137.7 MB)
  📄 epoch_03.pth  (137.7 MB)
  📄 epoch_04.pth  (137.7 MB)
  📄 epoch_05.pth  (137.7 MB)
  📄 epoch_06.pth  (137.7 MB)
  📄 epoch_07.pth  (137.7 MB)
  📄 epoch_08.pth  (137.7 MB)
  📄 epoch_09.pth  (137.7 MB)
  📄 epoch_10.pth  (137.7 MB)
  📄 epoch_11.pth  (137.7 MB)
  📄 epoch_12.pth  (137.7 MB)
  📄 epoch_13.pth  (137.7 MB)
  📄 epoch_14.pth  (137.7 MB)
  📄 epoch_15.pth  (137.7 MB)
  📄 epoch_16.pth  (137.7 MB)
  📄 epoch_17.pth  (137.7 MB)
  📄 epoch_18.pth  (137.7 MB)
  📄 epoch_19.pth  (137.7 MB)
  📄 epoch_20.pth  (137.7 MB)


In [ ]:
!ls "/content/drive/MyDrive/" | grep mini

v1.0-mini.tgz
v1 (1).0-mini.tgz


In [ ]:
import os
os.makedirs('/content/dataset', exist_ok=True)

# Try the first file
!tar -xzf "/content/drive/MyDrive/v1.0-mini.tgz" -C /content/dataset/

!ls /content/dataset/

LICENSE  maps  samples	sweeps	v1.0-mini


In [ ]:
# Use the exact filename shown above
import subprocess

result = subprocess.run([
    'tar', '-xzf',
    '/content/drive/MyDrive/v1.0-mini.tgz',
    '-C', '/content/dataset/'
], capture_output=True, text=True)

print("stdout:", result.stdout)
print("stderr:", result.stderr)
print("Return code:", result.returncode)

!ls /content/dataset/

stdout: 
stderr: 
Return code: 0
LICENSE  maps  samples	sweeps	v1.0-mini


In [ ]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(
    version  = 'v1.0-mini',
    dataroot = '/content/dataset',
    verbose  = False
)
print(f"Scenes : {len(nusc.scene)}")
print(f"Samples: {len(nusc.sample)}")
print("Dataset OK ✓")

Scenes : 10
Samples: 404
Dataset OK ✓


In [ ]:
import os, sys

os.chdir('/content/Bird-s-Eye-View-BEV-2D-Occupancy')
sys.path.insert(0, '/content/Bird-s-Eye-View-BEV-2D-Occupancy')

print("Files:", os.listdir('.'))

Files: ['train.py', 'logger', 'data', '.git', 'notebooks', 'config', 'requirements.txt', '.gitignore', 'eval.py', 'utils', 'models', 'exception', 'results']


In [ ]:
import config.config as cfg
cfg.DATAROOT = '/content/dataset'
print("DATAROOT:", cfg.DATAROOT)

DATAROOT: /content/dataset


In [ ]:
import torch
from data.nuscenes_loader import get_dataloaders
from models.bev_model import BEVOccupancyModel

device = torch.device('cuda')

train_loader, val_loader, ts, vs = get_dataloaders(
    dataroot='/content/dataset'
)

model = BEVOccupancyModel(pretrained=True).to(device)
batch = next(iter(train_loader))

out = model(
    batch['imgs'].to(device),
    batch['intrinsics'].to(device),
    batch['extrinsics'].to(device)
)
print(f"Output : {tuple(out['occ'].shape)}")
print(f"Device : {out['occ'].device}")
print("Model on T4 GPU ✓")

2026-03-22 14:31:22 | INFO     | data.nuscenes_loader | Dataset loaded | version: v1.0-mini | samples: 404
2026-03-22 14:31:22 | INFO     | data.nuscenes_loader | DataLoaders ready | train: 323 | val: 81


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s]
2026-03-22 14:31:23 | INFO     | models.backbone | ImageBackbone initialized | out_channels: 128 | pretrained: True
2026-03-22 14:31:23 | INFO     | models.lss_transformer | LSSViewTransformer initialized | D=64 bins [2.0m → 58.0m] | BEV=200×200 | res=0.4m/px
2026-03-22 14:31:23 | INFO     | models.bev_decoder | BEVDecoder initialized | in: 128 → out: 64 | upsamples 2× using 2D FCN (FastOcc)
2026-03-22 14:31:23 | INFO     | models.bev_decoder | OccupancyHead initialized | fuses BEV(64ch) + img(128ch) → 1 channel
2026-03-22 14:31:23 | INFO     | models.bev_model | BEVOccupancyModel ready | total params: 11,439,554 | trainable: 11,439,554
2026-03-22 14:31:24 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-22 14:31:24 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-22 14:31:24 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-22 14:31:24 | INFO     | data.preproce

Output : (2, 1, 200, 200)
Device : cuda:0
Model on T4 GPU ✓


In [ ]:
import os
import sys
import torch
import torch.optim as optim
from tqdm import tqdm
import shutil

os.chdir('/content/Bird-s-Eye-View-BEV-2D-Occupancy')
sys.path.insert(0, '/content/Bird-s-Eye-View-BEV-2D-Occupancy')

import config.config as cfg
cfg.DATAROOT = '/content/dataset'

from data.nuscenes_loader import get_dataloaders
from models.bev_model     import BEVOccupancyModel
from utils.metrics        import compute_metrics

device = torch.device('cuda')

# ── Drive save path ────────────────────────────────────
DRIVE_SAVE = '/content/drive/MyDrive/BEV_PROJECT'
os.makedirs(DRIVE_SAVE, exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Drive save path: {DRIVE_SAVE}")

# ── Data ───────────────────────────────────────────────
train_loader, val_loader, ts, vs = get_dataloaders(
    dataroot='/content/dataset'
)
print(f"Train: {ts} | Val: {vs}")

# ── Model ──────────────────────────────────────────────
# Check if checkpoint exists on Drive — resume training!
DRIVE_CKPT = f'{DRIVE_SAVE}/best_model.pth'

model     = BEVOccupancyModel(pretrained=True).to(device)
optimizer = optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20, eta_min=2e-6
)

# ── Resume from checkpoint if exists ──────────────────
start_epoch  = 1
best_iou     = 0.0
train_losses = []
val_ious     = []
val_dwes     = []

if os.path.exists(DRIVE_CKPT):
    print("Found checkpoint on Drive! Resuming...")
    ckpt = torch.load(DRIVE_CKPT, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch  = ckpt['epoch'] + 1
    best_iou     = ckpt['val_iou']
    train_losses = ckpt.get('train_losses', [])
    val_ious     = ckpt.get('val_ious', [])
    val_dwes     = ckpt.get('val_dwes', [])
    print(f"Resumed from epoch {ckpt['epoch']} | Best IoU: {best_iou:.4f}")
else:
    print("No checkpoint found — starting fresh")

# ── Training loop ──────────────────────────────────────
for epoch in range(start_epoch, 21):

    # Train
    model.train()
    epoch_loss  = 0.0
    num_batches = 0

    pbar = tqdm(
        train_loader,
        desc  = f"Epoch {epoch:02d}/20 [Train]",
        ncols = 80
    )

    for batch in pbar:
        imgs       = batch['imgs'].to(device)
        intrinsics = batch['intrinsics'].to(device)
        extrinsics = batch['extrinsics'].to(device)
        occ_gt     = batch['occ_gt'].to(device)

        outputs = model(imgs, intrinsics, extrinsics)
        losses  = model.compute_loss(outputs, occ_gt)
        loss    = losses['total']

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )
        optimizer.step()
        epoch_loss  += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    scheduler.step()

    # Validate
    model.eval()
    val_iou_sum = 0.0
    val_dwe_sum = 0.0
    val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            imgs       = batch['imgs'].to(device)
            intrinsics = batch['intrinsics'].to(device)
            extrinsics = batch['extrinsics'].to(device)
            occ_gt     = batch['occ_gt'].to(device)

            pred    = model(imgs, intrinsics, extrinsics)['occ']
            metrics = compute_metrics(
                pred, occ_gt.unsqueeze(1)
            )
            val_iou_sum += metrics['occ_iou']
            val_dwe_sum += metrics['dwe']
            val_batches += 1

    avg_iou = val_iou_sum / val_batches
    avg_dwe = val_dwe_sum / val_batches
    val_ious.append(avg_iou)
    val_dwes.append(avg_dwe)

    print(
        f"Epoch {epoch:02d}/20 | "
        f"Loss: {avg_loss:.4f} | "
        f"IoU: {avg_iou:.4f} | "
        f"DWE: {avg_dwe:.6f}"
    )

    # ── SAVE TO DRIVE EVERY EPOCH ──────────────────────
    # This runs after EVERY epoch
    # Session can crash anytime — data is safe on Drive!
    checkpoint = {
        'epoch'       : epoch,
        'model_state' : model.state_dict(),
        'optimizer'   : optimizer.state_dict(),
        'val_iou'     : avg_iou,
        'val_dwe'     : avg_dwe,
        'best_iou'    : best_iou,
        'train_losses': train_losses,
        'val_ious'    : val_ious,
        'val_dwes'    : val_dwes,
    }

    # Save latest epoch always
    torch.save(
        checkpoint,
        f'{DRIVE_SAVE}/epoch_{epoch:02d}.pth'
    )

    # Save best model separately
    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(
            checkpoint,
            f'{DRIVE_SAVE}/best_model.pth'
        )
        print(f"  ✓ Best model saved to Drive! IoU: {best_iou:.4f}")
    else:
        print(f"  ✓ Epoch {epoch} saved to Drive")

print(f"\nTraining complete! Best IoU: {best_iou:.4f}")
print(f"All checkpoints at: {DRIVE_SAVE}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Bird-s-Eye-View-BEV-2D-Occupancy'

In [17]:
import os, sys, shutil, torch

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results',     exist_ok=True)

# Copy from Drive
shutil.copytree(
    '/content/drive/MyDrive/BEV_PROJECT',
    'checkpoints',
    dirs_exist_ok=True
)

print("── Checkpoint Analysis ─────────────")
best_iou  = 0
best_file = None

files = sorted([
    f for f in os.listdir('checkpoints')
    if f.endswith('.pth')
])

for f in files:
    ckpt    = torch.load(f'checkpoints/{f}', map_location='cpu')
    epoch   = ckpt.get('epoch',      '?')
    val_iou = ckpt.get('val_iou',     0)
    val_dwe = ckpt.get('val_dwe',     0)
    loss    = ckpt.get('train_loss',  0)

    print(
        f"{f:20s} | "
        f"epoch:{epoch:2} | "
        f"IoU:{val_iou:.4f} | "
        f"DWE:{val_dwe:.4f}"
    )

    if val_iou > best_iou:
        best_iou  = val_iou
        best_file = f

print()
print(f"Best file : {best_file}")
print(f"Best IoU  : {best_iou:.4f}")

── Checkpoint Analysis ─────────────
best_model.pth       | epoch: 7 | IoU:0.2148 | DWE:0.3909
epoch_01.pth         | epoch: 1 | IoU:0.0515 | DWE:0.5530
epoch_02.pth         | epoch: 2 | IoU:0.1386 | DWE:0.5526
epoch_03.pth         | epoch: 3 | IoU:0.1516 | DWE:0.5006
epoch_04.pth         | epoch: 4 | IoU:0.1727 | DWE:0.5969
epoch_05.pth         | epoch: 5 | IoU:0.1558 | DWE:0.6451
epoch_06.pth         | epoch: 6 | IoU:0.1458 | DWE:0.6559
epoch_07.pth         | epoch: 7 | IoU:0.2148 | DWE:0.3909
epoch_08.pth         | epoch: 8 | IoU:0.1860 | DWE:0.4276
epoch_09.pth         | epoch: 9 | IoU:0.0910 | DWE:0.6059
epoch_10.pth         | epoch:10 | IoU:0.1092 | DWE:0.6249
epoch_11.pth         | epoch:11 | IoU:0.1160 | DWE:0.5542
epoch_12.pth         | epoch:12 | IoU:0.1158 | DWE:0.4875
epoch_13.pth         | epoch:13 | IoU:0.1468 | DWE:0.5046
epoch_14.pth         | epoch:14 | IoU:0.1722 | DWE:0.4886
epoch_15.pth         | epoch:15 | IoU:0.1383 | DWE:0.6735
epoch_16.pth         | epoch:16 | I

In [18]:
os.makedirs('/content/dataset', exist_ok=True)
!tar -xzf "/content/drive/MyDrive/v1.0-mini.tgz" -C /content/dataset/
print("Dataset:", os.listdir('/content/dataset'))

Dataset: ['maps', 'sweeps', '.v1.0-mini.txt', 'LICENSE', 'samples', 'v1.0-mini']


In [19]:
import config.config as cfg
cfg.DATAROOT = '/content/dataset'

from nuscenes.nuscenes import NuScenes
nusc = NuScenes(
    version  = 'v1.0-mini',
    dataroot = cfg.DATAROOT,
    verbose  = False
)
print(f"Scenes : {len(nusc.scene)}")
print(f"Samples: {len(nusc.sample)}")
print("Dataset OK ✓")

Scenes : 10
Samples: 404
Dataset OK ✓


In [23]:
!git fetch origin
!git reset --hard origin/main
print("Code reset ✓")

HEAD is now at e8d9c09 Initial clean commit (removed large files)
Code reset ✓


In [24]:
!grep -n "plot_before_after_training" utils/visualize.py

402:def plot_before_after_training(


In [26]:
# Force reload the module
import importlib
import utils.visualize
importlib.reload(utils.visualize)

from utils.visualize import plot_before_after_training
print("Import OK ✓")

Import OK ✓


In [27]:
import torch
import matplotlib
matplotlib.use('Agg')
import os

device = torch.device('cuda')

from models.bev_model     import BEVOccupancyModel
from data.nuscenes_loader import get_dataloaders
from utils.metrics        import compute_metrics
from utils.visualize      import plot_before_after_training
import config.config as cfg

cfg.DATAROOT = '/content/dataset'

# Untrained model
model_before = BEVOccupancyModel(pretrained=False).to(device)
model_before.eval()

# Best trained model
model_after = BEVOccupancyModel(pretrained=False).to(device)
ckpt = torch.load(
    'checkpoints/best_model.pth',
    map_location=device
)
model_after.load_state_dict(ckpt['model_state'])
model_after.eval()

print(f"Best model | epoch: {ckpt['epoch']} | IoU: {ckpt['val_iou']:.4f}")

# Get data
_, val_loader, _, _ = get_dataloaders(
    dataroot='/content/dataset'
)

# Generate 5 comparisons
os.makedirs('results', exist_ok=True)

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 5:
            break

        imgs       = batch['imgs'].to(device)
        intrinsics = batch['intrinsics'].to(device)
        extrinsics = batch['extrinsics'].to(device)
        gt         = batch['occ_gt']
        gt_gpu     = gt.unsqueeze(1).to(device)

        pred_before = model_before(
            imgs, intrinsics, extrinsics
        )['occ']
        pred_after  = model_after(
            imgs, intrinsics, extrinsics
        )['occ']

        m_before = compute_metrics(pred_before, gt_gpu)
        m_after  = compute_metrics(pred_after,  gt_gpu)

        print(
            f"Sample {i} | "
            f"Before: {m_before['occ_iou']:.4f} | "
            f"After : {m_after['occ_iou']:.4f} | "
            f"+{m_after['occ_iou']-m_before['occ_iou']:.4f}"
        )

        plot_before_after_training(
            imgs           = batch['imgs'][0],
            pred_before    = pred_before[0].cpu(),
            pred_after     = pred_after[0].cpu(),
            gt             = gt[0],
            metrics_before = m_before,
            metrics_after  = m_after,
            save_path      = f'results/before_after_{i}.png',
            sample_id      = i
        )

print("\nAll visualizations saved ✓")

2026-03-23 11:47:52 | INFO     | models.backbone | ImageBackbone initialized | out_channels: 128 | pretrained: False
2026-03-23 11:47:52 | INFO     | models.lss_transformer | LSSViewTransformer initialized | D=64 bins [2.0m → 58.0m] | BEV=200×200 | res=0.4m/px
2026-03-23 11:47:52 | INFO     | models.bev_decoder | BEVDecoder initialized | in: 128 → out: 64 | upsamples 2× using 2D FCN (FastOcc)
2026-03-23 11:47:52 | INFO     | models.bev_decoder | OccupancyHead initialized | fuses BEV(64ch) + img(128ch) → 1 channel
2026-03-23 11:47:52 | INFO     | models.bev_model | BEVOccupancyModel ready | total params: 11,439,554 | trainable: 11,439,554
2026-03-23 11:47:53 | INFO     | models.backbone | ImageBackbone initialized | out_channels: 128 | pretrained: False
2026-03-23 11:47:53 | INFO     | models.lss_transformer | LSSViewTransformer initialized | D=64 bins [2.0m → 58.0m] | BEV=200×200 | res=0.4m/px
2026-03-23 11:47:53 | INFO     | models.bev_decoder | BEVDecoder initialized | in: 128 → out:

Best model | epoch: 7 | IoU: 0.2148


2026-03-23 11:47:54 | INFO     | data.nuscenes_loader | Dataset loaded | version: v1.0-mini | samples: 404
2026-03-23 11:47:54 | INFO     | data.nuscenes_loader | DataLoaders ready | train: 323 | val: 81
2026-03-23 11:47:54 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:47:54 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:47:54 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:47:54 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:47:54 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:47:54 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:47:54 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:47:54 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:47:54 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:47:54 | INF

Sample 0 | Before: 0.0000 | After : 0.2263 | +0.2263


2026-03-23 11:48:00 | INFO     | utils.visualize | Before/After saved: results/before_after_0.png
2026-03-23 11:48:00 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:00 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:00 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:00 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:00 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:00 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:00 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:00 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:00 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:00 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:00 | INFO     | data.

...

Sample 3 | Before: 0.0000 | After : 0.1949 | +0.1949


2026-03-23 11:48:11 | INFO     | utils.visualize | Before/After saved: results/before_after_3.png
2026-03-23 11:48:12 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:12 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:12 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:12 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:12 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:12 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:12 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:12 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:12 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:12 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:12 | INFO     | data.

Sample 4 | Before: 0.0000 | After : 0.2264 | +0.2264


2026-03-23 11:48:15 | INFO     | utils.visualize | Before/After saved: results/before_after_4.png
2026-03-23 11:48:15 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:15 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:15 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:15 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:15 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:15 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:15 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:15 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:48:15 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:48:15 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:48:15 | INFO     | data.


All visualizations saved ✓


In [28]:
import shutil, os
from google.colab import files

# Save to Drive
shutil.copytree(
    'results',
    '/content/drive/MyDrive/BEV_PROJECT/results',
    dirs_exist_ok=True
)
print("Saved to Drive ✓")

# Download all PNGs
for f in sorted(os.listdir('results')):
    if f.endswith('.png'):
        print(f"Downloading: {f}")
        files.download(f'results/{f}')

Saved to Drive ✓
Downloading: before_after_0.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: before_after_1.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: test_curves.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
# Cell — Full evaluation on validation set
import torch
import os
from tqdm import tqdm
import config.config as cfg

cfg.DATAROOT = '/content/dataset'
device = torch.device('cuda')

from models.bev_model     import BEVOccupancyModel
from data.nuscenes_loader import get_dataloaders
from utils.metrics        import compute_metrics
from utils.visualize      import plot_full_results

# Load best model
ckpt  = torch.load('checkpoints/best_model.pth', map_location=device)
model = BEVOccupancyModel(pretrained=False).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f"Loaded epoch {ckpt['epoch']} | IoU: {ckpt['val_iou']:.4f}")

# Get validation data
_, val_loader, _, val_size = get_dataloaders(
    dataroot='/content/dataset'
)
print(f"Evaluating on {val_size} samples...")

# Evaluate
total_iou = 0.0
total_dwe = 0.0
n_batches = 0
all_ious  = []
all_dwes  = []

with torch.no_grad():
    for i, batch in enumerate(tqdm(val_loader, desc="Evaluating")):
        imgs       = batch['imgs'].to(device)
        intrinsics = batch['intrinsics'].to(device)
        extrinsics = batch['extrinsics'].to(device)
        occ_gt     = batch['occ_gt'].to(device)

        outputs  = model(imgs, intrinsics, extrinsics)
        pred     = outputs['occ']
        gt       = occ_gt.unsqueeze(1)

        metrics    = compute_metrics(pred, gt)
        total_iou += metrics['occ_iou']
        total_dwe += metrics['dwe']
        n_batches += 1

        all_ious.append(metrics['occ_iou'])
        all_dwes.append(metrics['dwe'])

        # Save first 3 result plots
        if i < 3:
            plot_full_results(
                imgs      = batch['imgs'][0],
                pred      = pred[0].cpu(),
                gt        = occ_gt[0].cpu(),
                metrics   = metrics,
                save_path = f'results/eval_sample_{i}.png',
                sample_id = i
            )

# Final results
final_iou = total_iou / n_batches
final_dwe = total_dwe / n_batches

print()
print("═" * 50)
print("FINAL EVALUATION RESULTS")
print("═" * 50)
print(f"Occupancy IoU         : {final_iou:.4f}")
print(f"Distance-Weighted Err : {final_dwe:.6f}")
print(f"Min IoU               : {min(all_ious):.4f}")
print(f"Max IoU               : {max(all_ious):.4f}")
print(f"Avg IoU               : {final_iou:.4f}")
print("═" * 50)


2026-03-23 11:55:41 | INFO     | models.backbone | ImageBackbone initialized | out_channels: 128 | pretrained: False
2026-03-23 11:55:41 | INFO     | models.lss_transformer | LSSViewTransformer initialized | D=64 bins [2.0m → 58.0m] | BEV=200×200 | res=0.4m/px
2026-03-23 11:55:41 | INFO     | models.bev_decoder | BEVDecoder initialized | in: 128 → out: 64 | upsamples 2× using 2D FCN (FastOcc)
2026-03-23 11:55:41 | INFO     | models.bev_decoder | OccupancyHead initialized | fuses BEV(64ch) + img(128ch) → 1 channel
2026-03-23 11:55:41 | INFO     | models.bev_model | BEVOccupancyModel ready | total params: 11,439,554 | trainable: 11,439,554


Loaded epoch 7 | IoU: 0.2148


2026-03-23 11:55:42 | INFO     | data.nuscenes_loader | Dataset loaded | version: v1.0-mini | samples: 404
2026-03-23 11:55:42 | INFO     | data.nuscenes_loader | DataLoaders ready | train: 323 | val: 81


Evaluating on 81 samples...


Evaluating:   0%|          | 0/41 [00:00<?, ?it/s]2026-03-23 11:55:42 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:55:42 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:55:42 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:55:42 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:55:42 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:55:42 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:55:42 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:55:42 | INFO     | data.preprocess | Intrinsic K scaled for resized image
2026-03-23 11:55:42 | INFO     | data.preprocess | Extrinsic E matrix built
2026-03-23 11:55:42 | INFO     | data.preprocess | Image preprocessed | shape: (3, 256, 704)
2026-03-23 11:55:42 | INFO     | data.preprocess | Intrinsic K scaled for resized imag


══════════════════════════════════════════════════
FINAL EVALUATION RESULTS
══════════════════════════════════════════════════
Occupancy IoU         : 0.2161
Distance-Weighted Err : 0.391353
Min IoU               : 0.1771
Max IoU               : 0.2631
Avg IoU               : 0.2161
══════════════════════════════════════════════════


In [30]:
import shutil
from google.colab import files

# Save to Drive
shutil.copytree(
    'results',
    '/content/drive/MyDrive/BEV_PROJECT/results',
    dirs_exist_ok=True
)
print("Saved ✓")

# Download eval plots
for f in sorted(os.listdir('results')):
    if f.startswith('eval_') and f.endswith('.png'):
        files.download(f'results/{f}')


Saved ✓


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>